In [ ]:
def fitline(spectrum, lambda1, lambda2): 
    '''
    Fit a single emission line that is in the range (lambda1, lambda2) 
    '''
    sub_region = SpectralRegion(lambda1*u.Angstrom, lambda2*u.Angstrom)
    sub_spectrum = extract_region(spectrum, sub_region)
    line  = estimate_line_parameters(sub_spectrum, models.Gaussian1D())
    return line

In [ ]:
def fit_all_lines(subspec, line_windows, **kwargs):
    line_ic = []
    for window in line_windows:
        line_fit = fitline(subspec, window[0], window[1])
        if line_fit.mean > window[1]*u.angstrom:
            line_fit.mean = window[1]*u.angstrom
        if line_fit.mean < window[0]*u.angstrom:
            line_fit.mean = window[0]*u.angstrom
            
        line_ic.append(models.Gaussian1D(amplitude=line_fit.amplitude, mean=line_fit.mean, stddev=line_fit.stddev))
        print(line_fit)

    
    full_fit = fit_lines(subspec, line_ic, window=line_windows*u.angstrom)

    return full_fit

In [ ]:
def plot_line_models(subspec, line_fits):
    xrange = subspec.spectral_axis[[0, -1]]
    plt.plot(subspec.spectral_axis, subspec.flux, lw=3, color="black")
    plt.fill_between(subspec.spectral_axis.value, subspec.flux.value - subspec.uncertainty.array, subspec.flux.value + subspec.uncertainty.array,
                    alpha = 0.2)
        
    x = np.linspace(xrange[0], xrange[1], 1000)

    for linefit_ob in line_fits:
        model = (linefit_ob(x))
        plt.plot(x, model, alpha=1, lw=2)
    


In [ ]:
def contsub(spectrum, lambda1, lambda2, lambda3, lambda4):
    '''
    Calculate the continuum in regions (lambda1, lambda2) and (lambda3, lambda4)
    Subtract the continuum and return a new spectrum for just the range (lambda1, lambda4) 
    '''
    region = [(lambda1 * u.Angstrom, lambda2 * u.Angstrom), (lambda3 * u.Angstrom, lambda4 * u.Angstrom)]
    fitted_continuum = fit_continuum(spec2, window=region)
    contfit = fitted_continuum(spec2.wavelength)
    new_spec = desi_utils.Spectrum(flux=spec2.flux-contfit, wavelength=spec2.spectral_axis, uncertainty=spec2.uncertainty)
    
    return new_spec


In [ ]:
def plot_line_fit(subspec, models, line, width=25):
    λ0 = desi_utils.retrieve_wavelength(line) 
    model_idx = np.argmin([np.abs(model.mean.value - λ0) for model in models])

    model = models[model_idx]

    xrange = (λ0 - width, λ0 + width)
    plot_spectrum(subspec, lw=3, color="black")
    plt.fill_between(subspec.spectral_axis.value, subspec.flux.value - subspec.uncertainty.array, subspec.flux.value + subspec.uncertainty.array,
                    alpha = 0.2, color="black")


    xmodel = np.linspace(*xrange, 1000)
    ymodel = model(xmodel * u.angstrom)
    plt.plot(xmodel, ymodel, alpha=1, lw=2)


    # plot catalogue model
    model_idx = np.argmin([np.abs(model.mean.value - λ0) for model in models])

    model = models[model_idx]

    
    plt.xlim(*xrange)
    plt.title(line)
